# 03 — Batch & Composite Experiments

`QickworkspaceV2` provides two composite helpers:

| Function | Behaviour |
|---|---|
| `run_batch` | Runs experiments **sequentially**, in order.  Each can feed results into the next. |
| `run_parallel` | Runs experiments **in parallel** (separate threads). Use only with independent hardware sessions. |

Both return a `dict[name → ExperimentData]`.

In [ ]:
import sys; sys.path.insert(0, '../')
from qick.asm_v2 import QickSweep1D

from QickworkspaceV2 import BaseExperiment, ExperimentConfig, run_batch, run_parallel
from QickworkspaceV2.config.system_cfg import config_list
from QickworkspaceV2.experiments.resonator import ResonatorSpec
from QickworkspaceV2.experiments.qubit_ge import QubitSpec, PowerRabi
from QickworkspaceV2.experiments.coherence import T1, Ramsey

BaseExperiment.connect_pyro4(
    ns_host='192.168.10.82', ns_port=8888, proxy_name='myqick',
    data_path=r'D:\Labber_Data\Jay\test',
)

qubit = 'Q1'
cfg_all = ExperimentConfig(config_list)

res_center = cfg_all.get_qubit(qubit)['res_freq_ge']
res_cfg = cfg_all.get_qubit(qubit)
res_cfg.update([('steps', 51), ('res_freq_ge', QickSweep1D('freqloop', res_center-10, res_center+10)), ('relax_delay', 0)])

qb_center = cfg_all.get_qubit(qubit)['qb_freq_ge']
qb_cfg = cfg_all.get_qubit(qubit)
qb_cfg.update([('steps', 51), ('qb_freq_ge', QickSweep1D('freqloop', qb_center-50, qb_center+50)), ('qb_mixer', qb_center), ('qb_gain_ge', 0.1), ('qb_flat_top_length_ge', 1.0)])

rabi_cfg = cfg_all.get_qubit(qubit)
rabi_cfg.update([('steps', 51), ('qb_gain_ge', QickSweep1D('gainloop', 0.0, 1.0))])


## run_batch — sequential pipeline

Each entry is `(name, experiment_instance)`.  Results are available immediately
after the batch completes as `results[name]`.

In [ ]:
experiments = [
    ('res_spec', ResonatorSpec(res_cfg)),
    ('qubit_spec', QubitSpec(qb_cfg)),
    ('power_rabi', PowerRabi(rabi_cfg)),
]

results = run_batch(experiments, py_avg=5)
print('Completed steps:', list(results.keys()))


In [ ]:
# Access individual results by name
for name, r in results.items():
    val = f'{r.scalar_result:.4f}' if r.scalar_result is not None else 'N/A'
    print(f'{name:<15} quality={r.quality.value:<12} scalar={val}')

## Feeding results between steps

The cleanest pattern: update `cfg_all` after each step so later steps use
the freshly calibrated values.

In [ ]:
from QickworkspaceV2 import CalibrationStore
import tempfile, os

store = CalibrationStore(os.path.join(tempfile.gettempdir(), 'batch_demo.json'))

def update(key, result, result_key=None):
    val = result.get_param(result_key) if result_key else result.scalar_result
    if val is not None:
        cfg_all.update(key, val, q_index=qubit)
        store.set(qubit, key, val)
        print(f'  updated {key} = {val:.4f}')

center = cfg_all.get_qubit(qubit)['res_freq_ge']
run_cfg = cfg_all.get_qubit(qubit)
run_cfg.update([('steps', 51), ('res_freq_ge', QickSweep1D('freqloop', center-10, center+10)), ('relax_delay', 0)])
r = ResonatorSpec(run_cfg).run(py_avg=5)
update('res_freq_ge', r)

center = cfg_all.get_qubit(qubit)['qb_freq_ge']
run_cfg = cfg_all.get_qubit(qubit)
run_cfg.update([('steps', 51), ('qb_freq_ge', QickSweep1D('freqloop', center-50, center+50)), ('qb_mixer', center), ('qb_gain_ge', 0.1), ('qb_flat_top_length_ge', 1.0)])
r = QubitSpec(run_cfg).run(py_avg=5)
update('qb_freq_ge', r)

run_cfg = cfg_all.get_qubit(qubit)
run_cfg.update([('steps', 51), ('qb_gain_ge', QickSweep1D('gainloop', 0.0, 1.0))])
r = PowerRabi(run_cfg).run(py_avg=5)
update('pi_gain_ge', r, 'pi_gain')
update('pi2_gain_ge', r, 'pi2_gain')

print(store.summary(qubit))


## run_parallel — use only when concurrent hardware access is safe

`run_parallel` uses Python threads. The current framework does not provide a hardware queue,
so do not run multiple experiments concurrently against the same QICK board. The following cell
shows the call but leaves execution commented out.


In [ ]:
t1_cfg = cfg_all.get_qubit(qubit)
t1_cfg.update([('steps', 51), ('wait_time', QickSweep1D('waitloop', 0.0, 150.0))])

ramsey_cfg = cfg_all.get_qubit(qubit)
ramsey_cfg.update([('steps', 51), ('wait_time', QickSweep1D('waitloop', 0.0, 5.0)), ('virtual_detune', 1.0)])

# Run only if these experiments do not share one hardware resource:
# par_results = run_parallel([
#     ('t1', T1(t1_cfg)),
#     ('ramsey', Ramsey(ramsey_cfg)),
# ], py_avg=5)


**Next:** [04_auto_calibrate.ipynb](04_auto_calibrate.ipynb) — fully automated calibration with `AutoCalibrate`.